# MS COCO — Multi-label classification challenge

Ce notebook démarre le projet demandé dans le notebook du cours :
- transfer learning sur le dataset MS COCO fourni ;
- classification multi-label sur 80 classes ;
- séparation train/validation reproductible ;
- sauvegarde du meilleur modèle selon le F1 de validation ;
- suivi des expériences et des coûts de calcul ;
- génération du JSON de soumission.

**Important :** configure `DATA_ROOT` vers le dossier `ms-coco` fourni par le cours. Ne télécharge pas une autre version de MS COCO : le sujet l'interdit.

## 1. Configuration

Structure attendue, d'après le notebook du cours :

```text
ms-coco/
├── images/
│   ├── train et test mélangés (fichiers .jpg)
└── labels/
    └── fichiers .cls pour les images d'entraînement
```

Le code ci-dessous suppose que les dossiers sont `ms-coco/images` et `ms-coco/labels`. Adapte `DATA_ROOT` si ta clé USB utilise un autre chemin.

In [ ]:
from pathlib import Path
import os, json, time, random, platform, csv
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

SEED = 42
DATA_ROOT = Path("ms-coco")  # À MODIFIER si nécessaire
IMAGES_DIR = DATA_ROOT / "images"
LABELS_DIR = DATA_ROOT / "labels"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.2
NUM_WORKERS = 2
THRESHOLD = 0.5

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "best_resnet18.pth"
EXPERIMENTS_CSV = OUTPUT_DIR / "experiments.csv"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert IMAGES_DIR.exists(), f"Dossier images introuvable : {IMAGES_DIR.resolve()}"
assert LABELS_DIR.exists(), f"Dossier labels introuvable : {LABELS_DIR.resolve()}"

## 2. Les 80 classes

L'ordre est celui du notebook du cours. Les indices des labels dans les fichiers `.cls` correspondent à cette liste.

In [ ]:
CLASSES = (
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush"
)
NUM_CLASSES = len(CLASSES)
assert NUM_CLASSES == 80

## 3. Dataset et transformations

Chaque image est associée à un vecteur multi-hot de 80 valeurs :
- `1` si la classe est présente ;
- `0` sinon.

Pour éviter que l'augmentation aléatoire s'applique à la validation, les transformations sont appliquées dans un wrapper distinct après la séparation train/validation.

In [ ]:
class COCOTrainBase(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.label_files = sorted(self.labels_dir.glob("*.cls"))
        if not self.label_files:
            raise RuntimeError(f"Aucun fichier .cls trouvé dans {self.labels_dir.resolve()}")

    def __len__(self):
        return len(self.label_files)

    def __getitem__(self, idx):
        label_path = self.label_files[idx]
        image_path = self.images_dir / f"{label_path.stem}.jpg"
        if not image_path.exists():
            raise FileNotFoundError(f"Image correspondante introuvable : {image_path}")
        image = Image.open(image_path).convert("RGB")
        with open(label_path, "r", encoding="utf-8") as f:
            class_ids = [int(line.strip()) for line in f if line.strip()]
        labels = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        if class_ids:
            if min(class_ids) < 0 or max(class_ids) >= NUM_CLASSES:
                raise ValueError(f"ID de classe hors limites dans {label_path}")
            labels[class_ids] = 1.0
        return image, labels, label_path.stem


class TransformSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, labels, image_id = self.base_dataset[self.indices[idx]]
        if self.transform:
            image = self.transform(image)
        return image, labels, image_id


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

base_dataset = COCOTrainBase(IMAGES_DIR, LABELS_DIR)
n_total = len(base_dataset)
n_val = max(1, int(n_total * VAL_FRACTION))
n_train = n_total - n_val

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(n_total, generator=generator).tolist()
train_indices = permutation[:n_train]
val_indices = permutation[n_train:]

train_dataset = TransformSubset(base_dataset, train_indices, train_transform)
val_dataset = TransformSubset(base_dataset, val_indices, val_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

print(f"Images annotées : {n_total:,}")
print(f"Train : {len(train_dataset):,} | Validation : {len(val_dataset):,}")

## 4. Vérification rapide des données

Exécute cette cellule avant de lancer l'entraînement. Elle vérifie les dimensions, le type de labels et le nombre moyen de classes par image.

In [ ]:
images_batch, labels_batch, ids_batch = next(iter(train_loader))
print("Batch images :", tuple(images_batch.shape))
print("Batch labels :", tuple(labels_batch.shape))
print("Type images :", images_batch.dtype)
print("Type labels :", labels_batch.dtype)
print("Nombre moyen de classes par image dans ce batch :", labels_batch.sum(dim=1).mean().item())
print("Exemples d'identifiants :", ids_batch[:5])
assert images_batch.ndim == 4 and images_batch.shape[1] == 3
assert labels_batch.shape[1] == NUM_CLASSES

## 5. Modèle baseline : ResNet18 pré-entraîné

On remplace la tête de classification par une couche à 80 sorties. Le modèle renvoie des **logits** : on utilise donc `BCEWithLogitsLoss`, sans ajouter de sigmoid dans le modèle. La sigmoid sera appliquée pour calculer les probabilités et les prédictions.

Si le téléchargement des poids pré-entraînés échoue, vérifie l'accès Internet de l'environnement. Le sujet demande une stratégie de transfer learning.

In [ ]:
def build_resnet18(num_classes=NUM_CLASSES, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_resnet18(pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

print(f"Paramètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Métriques multi-label

On calcule les métriques à partir des probabilités `sigmoid(logits)` et d'un seuil de probabilité. Le F1 macro donne le même poids à chaque classe ; le F1 micro agrège les décisions de toutes les classes. On enregistre les deux pour mieux comprendre les résultats.

Le score local peut différer du score du serveur de cours : le notebook fourni décrit une évaluation moyenne par classe, mais le détail exact du calcul du serveur n'est pas entièrement spécifié dans le notebook.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.5):
    model.eval()
    total_loss = 0.0
    all_probs, all_targets = [], []

    for images, targets, _ in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, targets)
        total_loss += loss.item() * images.size(0)
        all_probs.append(torch.sigmoid(logits).cpu())
        all_targets.append(targets.cpu())

    probs = torch.cat(all_probs)
    targets = torch.cat(all_targets).bool()
    preds = probs >= threshold

    tp = (preds & targets).sum(dim=0).float()
    fp = (preds & ~targets).sum(dim=0).float()
    fn = (~preds & targets).sum(dim=0).float()

    class_precision = tp / (tp + fp).clamp_min(1)
    class_recall = tp / (tp + fn).clamp_min(1)
    class_f1 = 2 * class_precision * class_recall / (class_precision + class_recall).clamp_min(1e-12)

    micro_tp = tp.sum()
    micro_fp = fp.sum()
    micro_fn = fn.sum()
    micro_precision = micro_tp / (micro_tp + micro_fp).clamp_min(1)
    micro_recall = micro_tp / (micro_tp + micro_fn).clamp_min(1)
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall).clamp_min(1e-12)

    return {
        "loss": total_loss / len(loader.dataset),
        "macro_precision": class_precision.mean().item(),
        "macro_recall": class_recall.mean().item(),
        "macro_f1": class_f1.mean().item(),
        "micro_precision": micro_precision.item(),
        "micro_recall": micro_recall.item(),
        "micro_f1": micro_f1.item(),
        "class_f1": class_f1.numpy(),
    }


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    n = 0
    for images, targets, _ in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / max(n, 1)

## 7. Entraînement baseline

La sélection du meilleur checkpoint est faite uniquement sur le F1 macro de validation. Les données de test ne sont pas utilisées pour choisir les hyperparamètres.

In [ ]:
history = []
best_val_f1 = -1.0
best_epoch = None

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

start_training = time.perf_counter()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    train_metrics = evaluate(model, train_loader, criterion, threshold=THRESHOLD)
    val_metrics = evaluate(model, val_loader, criterion, threshold=THRESHOLD)
    epoch_seconds = time.perf_counter() - epoch_start

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_precision": val_metrics["macro_precision"],
        "val_macro_recall": val_metrics["macro_recall"],
        "epoch_seconds": epoch_seconds,
        "threshold": THRESHOLD,
        "model": "ResNet18",
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss {train_loss:.4f} | val loss {val_metrics['loss']:.4f} | "
        f"val macro-F1 {val_metrics['macro_f1']:.4f} | "
        f"val micro-F1 {val_metrics['micro_f1']:.4f} | "
        f"{epoch_seconds:.1f}s"
    )

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        torch.save({
            "model_name": "resnet18",
            "model_state_dict": model.state_dict(),
            "image_size": IMAGE_SIZE,
            "threshold": THRESHOLD,
            "best_val_macro_f1": best_val_f1,
            "epoch": epoch,
            "seed": SEED,
        }, CHECKPOINT_PATH)

total_training_seconds = time.perf_counter() - start_training
pd.DataFrame(history).to_csv(OUTPUT_DIR / "baseline_history.csv", index=False)

peak_gpu_memory_mb = (
    torch.cuda.max_memory_allocated() / (1024**2) if device.type == "cuda" else None
)
print("\nEntraînement terminé.")
print("Meilleure époque :", best_epoch)
print("Meilleur F1 macro validation :", round(best_val_f1, 4))
print("Temps total (secondes) :", round(total_training_seconds, 1))
print("Pic mémoire GPU (MiB) :", None if peak_gpu_memory_mb is None else round(peak_gpu_memory_mb, 1))
print("Checkpoint :", CHECKPOINT_PATH.resolve())

## 8. Courbes d'apprentissage

Ces courbes aident à repérer un sous-apprentissage ou un surapprentissage.

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("Loss par époque")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_macro_f1"], marker="o", label="Train macro-F1")
plt.plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="Validation macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("F1 par époque")
plt.grid(True)
plt.legend()
plt.show()

## 9. Expériences suivantes : tableau de suivi

Ne lance pas une grande grille d'hyperparamètres avant d'avoir confirmé que la baseline fonctionne. Une fois la baseline terminée, utilise le tableau pour planifier des comparaisons contrôlées.

Commence par ne changer qu'une variable à la fois. Pour comparer les architectures, conserve la même résolution, le même split, le même nombre d'époques et les mêmes métriques.

In [ ]:
planned_experiments = pd.DataFrame([
    {"id": 1, "model": "ResNet18", "image_size": 224, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "baseline"},
    {"id": 2, "model": "MobileNetV3-Large", "image_size": 224, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "à faire"},
    {"id": 3, "model": "EfficientNet-B0", "image_size": 224, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "à faire"},
    {"id": 4, "model": "ResNet18", "image_size": 160, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "à faire"},
    {"id": 5, "model": "ResNet18", "image_size": 128, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "à faire"},
    {"id": 6, "model": "ResNet18", "image_size": 224, "batch_size": 32, "lr": 3e-4, "optimizer": "AdamW", "fine_tuning": "all", "status": "à faire"},
    {"id": 7, "model": "ResNet18", "image_size": 224, "batch_size": 32, "lr": 1e-4, "optimizer": "AdamW", "fine_tuning": "head only", "status": "à faire"},
])
display(planned_experiments)
planned_experiments.to_csv(OUTPUT_DIR / "planned_experiments.csv", index=False)

## 10. Recherche du seuil de décision

Le seuil se choisit sur la validation, pas sur le test. Cette cellule utilise les probabilités du checkpoint sélectionné pour comparer plusieurs seuils.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

@torch.no_grad()
def collect_validation_probabilities(model, loader):
    model.eval()
    probs_all, targets_all = [], []
    for images, targets, _ in loader:
        logits = model(images.to(device, non_blocking=True))
        probs_all.append(torch.sigmoid(logits).cpu())
        targets_all.append(targets.cpu())
    return torch.cat(probs_all), torch.cat(targets_all).bool()

val_probs, val_targets = collect_validation_probabilities(model, val_loader)

def macro_f1_from_probs(probs, targets, threshold):
    preds = probs >= threshold
    tp = (preds & targets).sum(dim=0).float()
    fp = (preds & ~targets).sum(dim=0).float()
    fn = (~preds & targets).sum(dim=0).float()
    p = tp / (tp + fp).clamp_min(1)
    r = tp / (tp + fn).clamp_min(1)
    f1 = 2 * p * r / (p + r).clamp_min(1e-12)
    return f1.mean().item()

threshold_results = []
for threshold in np.arange(0.1, 0.91, 0.05):
    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "macro_f1": macro_f1_from_probs(val_probs, val_targets, float(threshold))
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)
best_threshold = float(threshold_df.loc[threshold_df["macro_f1"].idxmax(), "threshold"])
print("Meilleur seuil global sur validation :", best_threshold)
threshold_df.to_csv(OUTPUT_DIR / "threshold_search.csv", index=False)

plt.figure(figsize=(8, 4))
plt.plot(threshold_df["threshold"], threshold_df["macro_f1"], marker="o")
plt.xlabel("Seuil de probabilité")
plt.ylabel("Macro-F1 validation")
plt.title("Recherche du seuil")
plt.grid(True)
plt.show()

## 11. Génération du JSON de soumission

À exécuter uniquement après avoir confirmé que le checkpoint est celui que tu souhaites soumettre. Le code parcourt les `.jpg` du dossier `images` qui ne possèdent pas de fichier `.cls` correspondant, ce qui suit l'organisation décrite dans le notebook du cours. Vérifie cette hypothèse sur ton dossier avant soumission.

In [ ]:
class COCOTestImages(Dataset):
    def __init__(self, images_dir, transform):
        self.images_dir = Path(images_dir)
        all_images = sorted(self.images_dir.glob("*.jpg"))
        label_stems = {p.stem for p in LABELS_DIR.glob("*.cls")}
        self.image_paths = [p for p in all_images if p.stem not in label_stems]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        return self.transform(image), path.stem


test_dataset = COCOTestImages(IMAGES_DIR, val_transform)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
print("Images test détectées :", len(test_dataset))

In [ ]:
@torch.no_grad()
def make_submission(model, loader, threshold, output_path):
    model.eval()
    predictions_by_id = {}
    for images, image_ids in loader:
        logits = model(images.to(device, non_blocking=True))
        probs = torch.sigmoid(logits).cpu()
        binary = probs >= threshold
        for i, image_id in enumerate(image_ids):
            predictions_by_id[image_id] = torch.where(binary[i])[0].tolist()

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(predictions_by_id, f, indent=2)

    return predictions_by_id

# Recharge le meilleur checkpoint avant l'inférence.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Utilise le seuil choisi sur validation.
submission_path = OUTPUT_DIR / "predictions_resnet18.json"
submission = make_submission(model, test_loader, best_threshold, submission_path)

print("JSON créé :", submission_path.resolve())
print("Nombre d'images prédites :", len(submission))
print("Exemple :", next(iter(submission.items())) if submission else "Aucune image détectée")

## 12. Checklist avant soumission

- [ ] `DATA_ROOT` pointe vers le dataset fourni par le cours.
- [ ] Les images et labels sont bien appariés.
- [ ] Le split train/validation est reproductible.
- [ ] Le modèle utilise 80 sorties et `BCEWithLogitsLoss`.
- [ ] Le seuil a été choisi uniquement sur validation.
- [ ] Le meilleur checkpoint est sauvegardé.
- [ ] Le JSON contient les noms d'images sans extension et des listes d'indices de classes.
- [ ] Le nombre d'images dans le JSON correspond au nombre d'images de test.
- [ ] Le notebook et les résultats sont sauvegardés pour le rendu Moodle.

**À noter :** la cellule de détection du test suppose que toutes les images sans fichier `.cls` sont des images de test. Vérifie ce point sur ton dataset fourni avant de soumettre le JSON.